# Project Findings Summary — Sequence Memory RNN, Prepared for Sep 1-2 Presentation

This notebook consolidates the findings from notebooks 01 through 012 into one narrative document.
It's meant to be read start to finish, no code needs to be run, the one code cell (Section 8) just
renders a summary chart from numbers already established in earlier notebooks.

**Project goal:** predict, from hippocampal LFP recordings, (1) whether a trial was InSeq or OutSeq, and
(2) which odor (A-E) was presented, benchmarked against the MaGNet paper's graph-based baseline
(~66-74% depending on rat), with an explicit interpretability requirement throughout.


## 1. Data Understanding (Notebooks 01-02)

- Behavioral (`bvr`), LFP, and spike (`spk`) data share one common time axis, no alignment work needed.
- The trial marker (`InSeqLog`) fires at the exact moment of odor onset ("Poke-In"), confirmed via a
  0-sample offset against the odor channel, not a later decision point.
- Odor labels are completely clean: every trial has exactly one active odor channel, zero ambiguous
  cases, across all 5 rats.
- **Sampling rate is NOT constant across rats** (714-1000 Hz depending on session) — this became
  important later.
- InSeq/OutSeq is naturally imbalanced (~90/10), by task design, not a data issue.


## 2. Baseline Models (Notebooks 03-05)

| Model | Feature type | InSeq/OutSeq | Odor Identity |
|---|---|---|---|
| GLM (logistic regression) | Raw voltage mean/std | 55.9% | 34.6% |
| RNN (unweighted loss) | Raw time series | 50.0% (majority collapse) | 17.9% (below chance) |
| RNN (class-weighted loss) | Raw time series | ~55% (fix confirmed working) | ~32% |
| GLM | Spectral (FFT band power) | 59.2% | 30.5% |

Chance levels: 50.0% (InSeq/OutSeq), 20.0% (Odor Identity, 5-way).

Key early lesson: an unweighted RNN loss function collapses to predicting the majority class given the
~90/10 imbalance, fixed by class-weighting the loss, matching what the GLM already did via
`class_weight='balanced'`.


## 3. Advisor Feedback Response (Notebooks 06-007)

Two pieces of feedback addressed:

1. **Window definition clarity**: windowing made fully explicit and inline (not hidden in an imported
   function), with real timestamps printed to verify "0ms" unambiguously means Poke-In.
2. **Frequency bands**: Theta redefined as 4-12 Hz (previously a narrower 4-8 Hz + separate 8-12 Hz
   alpha), High Gamma (80-150 Hz) added.

**A real bug was found and fixed while addressing feedback point 1**: the session's sampling rate is not
uniform (confirmed ~1000 Hz early in a session vs. a 714 Hz session-wide average for Mitt). Windows sized
using a single average rate did not reliably span the intended real-world duration everywhere in a
recording. Fixed by locating window edges from each trial's real timestamp directly
(`np.searchsorted`), then resampling to a consistent output length. Retraining with the fix showed
results were NOT meaningfully different for Mitt specifically (55.5% vs. the earlier 55.9%), but the fix
is necessary for correctness and mattered more once extended to all 5 rats (see Section 5).


## 4. Sliding-Window Discovery: The Fixed 500ms Window Was Wrong (Notebook 008)

Testing many window positions (250ms window, 25ms steps, -500ms to +1500ms relative to Poke-In) on
Mitt showed the fixed 500ms-starting-at-Poke-In window used everywhere before this was leaving real
accuracy on the table. Best window found: InSeq/OutSeq accuracy rose from 55.5% (fixed window) to a
single-run peak of 69.5% at +1475ms. Odor Identity peaked earlier (+1025ms) than InSeq/OutSeq, a
plausible pattern (the odor is present from the start of the trial; the InSeq/OutSeq judgment likely
requires more processing time).


## 5. Multi-Rat Validation (Notebooks 009-011)

Extending the sliding-window search to all 5 rats, and validating with repeated cross-validation (25
scores per window position instead of 5) and each rat's own safe search span (computed from that rat's
real minimum trial gap), produced the final, validated results:

| Rat | Validated InSeq/OutSeq peak | Best offset | Validated Odor Identity peak |
|---|---|---|---|
| Mitt | 75.9% | +2350ms | 38.1% |
| Barat | 79.8% | +1700ms | 43.3% |
| Stella | 79.8% | +1500ms | 47.4% |
| Superchris | 88.7% | +1000ms | 43.1% |
| Buchanan | 71.2% | +1700ms | 46.4% |

**Key findings from this validation process:**

- **3 of 5 rats reach or approach the ~85% InSeq/OutSeq target**, using a later window than originally
  assumed. This is the single most important result of the project so far.
- **Mitt, the rat every earlier notebook was built and tuned around, is actually the weakest performer**
  of the 5. Worth stating directly rather than treating Mitt as representative.
- **Odor Identity has no sharp temporal peak.** Best offsets are inconsistent in sign across rats even
  under rigorous repeated validation, this supports a real interpretation: odor identity is broadly,
  weakly decodable across a wide window, not concentrated at one moment, rather than a specific "best
  time" existing.
- **Superchris's notably higher raw signal amplitude does not trace to one outlier channel** (0 of 21
  channels exceed 2 standard deviations above the mean), supporting a genuinely stronger or differently
  calibrated recording rather than a single bad electrode.
- **Superchris has only 21 LFP channels, not 22** (channel `T11` is absent), most likely a single wire
  that didn't yield usable signal for that specific animal's implant (see notebook 012 for the
  cross-rat audit this prompted).


## 6. Cross-Rat Channel Audit (Notebook 012)

Confirms exactly which rats have which LFP channels, closing the gap left by the earlier structural
audit (notebook 02), which only checked behavioral channel names, not LFP channel identity or count.
See notebook 012 for the full per-rat channel table. This must be accounted for (reading each rat's
actual channel list, not assuming a fixed 22) before any model pools data across rats.


## 7. Chart: Validated Peak Accuracy vs. Target, All Rats


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rats = ['Mitt', 'Barat', 'Stella', 'Superchris', 'Buchanan']
inseq_peaks = [0.759, 0.798, 0.798, 0.887, 0.712]
odor_peaks = [0.381, 0.433, 0.474, 0.431, 0.464]

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(rats))
width = 0.35
bars1 = ax.bar(x - width/2, inseq_peaks, width, label='InSeq/OutSeq (chance=0.5)', color='#4C72B0')
bars2 = ax.bar(x + width/2, odor_peaks, width, label='Odor Identity (chance=0.2)', color='#DD8452')
ax.axhline(0.85, color='green', linestyle='--', alpha=0.6, label='target (~0.85)')
ax.set_xticks(x)
ax.set_xticklabels(rats)
ax.set_ylabel('Validated peak balanced accuracy')
ax.set_title('Final validated results, all 5 rats (repeated cross-validation)')
ax.legend()
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 8. Open Items (Not Yet Done)

- Multi-rat pooled training (session-based split), not yet built, deliberately deferred given the
  validation work took priority and the Sep 1-2 timeline.
- Permutation test to formally confirm the strongest result (Superchris, 88.7%) is statistically
  distinguishable from chance.
- RNN retraining with the validated window and updated bands (only the GLM has been retrained with
  these; the RNN baseline in notebook 04 still uses the original fixed 500ms window).
- Confirmation with the lab (Zoey/Keiland/Wonjae) on whether Superchris's missing channel and the
  general per-rat channel gaps reflect known hardware issues.

## 9. Recommended Presentation Narrative

1. Start with the project goal and the MaGNet benchmark.
2. Walk through the pipeline build (audit -> baseline -> spectral features), noting the interpretability
   emphasis throughout.
3. Present the two real bugs found and fixed (RNN class imbalance collapse, sampling-rate windowing
   error) as evidence of a careful, validated process, not just a final number.
4. Present the sliding-window finding as the project's central discovery: window choice mattered more
   than model complexity so far.
5. Present the validated, multi-rat results table (Section 5) as the headline result, with the honest
   caveats already noted (Mitt weakest, odor has no sharp peak, Superchris's channel count difference).
6. Close with the open items as the explicit near-term plan.
